# Etapa 5 — Avaliação de Desempenho

**Objetivo:** Comparar os cinco modelos treinados e entender onde cada um acerta e erra.

| Modelo | Filtro | F1-macro |
|--------|--------|:--------:|
| Random Forest | Gaussiano | 0.8716 |
| XGBoost | Sem filtro | 0.9065 |
| XGBoost | Estatístico | 0.9067 |
| **XGBoost** | **Gaussiano** | **0.9082** |
| CNN-1D (FCN) | Gaussiano | 0.6685 |

Todos avaliados com **GroupKFold(5) por instância** — sem vazamento entre poços.
As métricas são carregadas dos JSONs gerados pelos scripts de treinamento (sem re-treinar).

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from config import FIGURES_DIR, METRICS_DIR, WINDOW_CLASSES

print('Bibliotecas carregadas!')

# Carregar todas as métricas
metrics_raw = {
    'RF (Gaussiano)':       json.load(open(METRICS_DIR / 'rf_window_class_metrics.json')),
    'XGBoost (Sem filtro)': json.load(open(METRICS_DIR / 'xgboost_nofilter_metrics.json')),
    'XGBoost (Estatístico)':json.load(open(METRICS_DIR / 'xgboost_statistical_metrics.json')),
    'XGBoost (Gaussiano)':  json.load(open(METRICS_DIR / 'xgboost_window_class_metrics.json')),
    'CNN-1D (FCN)':         json.load(open(METRICS_DIR / 'cnn1d_metrics.json')),
}
print(f'Métricas carregadas: {list(metrics_raw.keys())}')

## 5.1 Comparação Global — Todos os Modelos

- **F1-macro:** trata todas as classes igualmente — penaliza quando o modelo ignora classes raras
- **F1-weighted:** média ponderada pelo número de janelas por classe (favorece classes frequentes)
- **Accuracy:** fração de janelas classificadas corretamente (distorcida pelo desbalanceamento)

In [ ]:
def get_global(m):
    """Extrai métricas globais independentemente da estrutura do JSON."""
    if 'metrics_concat_folds' in m:
        mc = m['metrics_concat_folds']
        return mc['f1_macro'], mc['f1_weighted'], mc['accuracy']
    return m['f1_macro'], m['f1_weighted'], m['accuracy']

rows = []
for name, m in metrics_raw.items():
    f1m, f1w, acc = get_global(m)
    rows.append({'Modelo': name, 'F1-macro': f1m, 'F1-weighted': f1w, 'Accuracy': acc})

df_cmp = pd.DataFrame(rows).set_index('Modelo').sort_values('F1-macro', ascending=False)
print(df_cmp.to_string(float_format='{:.4f}'.format))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if i == 0 else ('#95a5a6' if 'CNN' in df_cmp.index[i] else '#3498db')
          for i in range(len(df_cmp))]
df_cmp['F1-macro'].plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.6)
ax.set_title('F1-macro por Modelo — Estado Operacional (17 classes)', fontsize=12)
ax.set_ylabel('F1-macro')
ax.set_ylim(0.5, 1.0)
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(df_cmp['F1-macro']):
    ax.text(i, v + 0.004, f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.2 Nested CV — Validação do Método de Avaliação

O `RandomizedSearchCV` seleciona hiperparâmetros usando o mesmo conjunto de validação que avalia o modelo.
Isso pode criar um viés positivo — os hiperparâmetros foram "ajustados" para aquele conjunto.

**Nested CV 5×3** (300 fits) quantifica esse viés:
- Loop externo (5 folds): avalia o modelo final em dados nunca vistos
- Loop interno (3 folds): seleciona hiperparâmetros dentro do conjunto de treino

Se `delta = nested − flat ≈ 0`, a estratégia de flat CV é metodologicamente confiável.

In [ ]:
nested = json.load(open(METRICS_DIR / 'rf_nested_cv_results.json'))

flat_f1   = nested['flat_cv_reference']['f1_macro_cv']
nested_f1 = nested['mean_f1_macro']
nested_std = nested['std_f1_macro']
delta = nested['delta_vs_flat_cv']

print('=== Nested CV 5×3 — Random Forest ===')
print(f'F1 por fold externo: {nested["outer_fold_scores"]}')
print(f'\nFlat CV   : {flat_f1:.4f}')
print(f'Nested CV : {nested_f1:.4f} ± {nested_std:.4f}')
print(f'Delta     : {delta:+.4f} ({delta*100:+.2f} p.p.)')

print()
if abs(delta) < 0.02:
    print('✓ Viés de seleção < 2 p.p. — flat CV com GroupKFold é metodologicamente honesto.')
    print('  As comparações entre modelos feitas com flat CV são válidas.')
else:
    print('! Viés de seleção relevante — revisar a estratégia de avaliação.')

## 5.3 Desempenho por Classe — XGBoost vs RF vs CNN-1D

Classes com F1 baixo são candidatas à análise SHAP (Etapa 6).
A CNN-1D tende a falhar em transientes (101–109) — que dependem de `diff1_std`/`diff2_std` para separação.

In [ ]:
xgb_pc  = metrics_raw['XGBoost (Gaussiano)']['per_class']
rf_pc   = metrics_raw['RF (Gaussiano)']['per_class']
cnn_pc  = metrics_raw['CNN-1D (FCN)'].get('per_class_f1', {})

# Mapear nomes WINDOW_CLASSES → F1 da CNN (CNN usa nome, não número)
cnn_by_name = {v: cnn_pc.get(v, float('nan')) for v in WINDOW_CLASSES.values()}

rows_cls = []
for str_c, stats in sorted(xgb_pc.items(), key=lambda x: int(x[0])):
    c     = int(str_c)
    label = WINDOW_CLASSES.get(c, str(c))
    f1_rf  = rf_pc.get(str_c, {}).get('f1', float('nan'))
    f1_xgb = stats['f1']
    f1_cnn = cnn_by_name.get(label, float('nan'))
    rows_cls.append({'Classe': c, 'Nome': label[:30],
                     'RF': f1_rf, 'XGBoost': f1_xgb, 'CNN-1D': f1_cnn,
                     'Suporte': stats.get('support', '')})

df_cls = pd.DataFrame(rows_cls).set_index('Classe')
print(df_cls.to_string(float_format='{:.3f}'.format))

best_c  = df_cls['XGBoost'].idxmax()
worst_c = df_cls['XGBoost'].idxmin()
print(f'\nMelhor  XGBoost: classe {best_c} — {df_cls.loc[best_c,"Nome"]}  (F1={df_cls.loc[best_c,"XGBoost"]:.3f})')
print(f'Pior    XGBoost: classe {worst_c} — {df_cls.loc[worst_c,"Nome"]}  (F1={df_cls.loc[worst_c,"XGBoost"]:.3f})')

## 5.4 Matrizes de Confusão

Células na diagonal = acertos. Células fora da diagonal = confusões entre estados.
Confusões frequentes entre classes semanticamente próximas são esperadas (ex: transiente vs ativo do mesmo evento).

In [ ]:
conf_dir = FIGURES_DIR / 'confusion_matrix'
conf_files = {
    'RF (Gaussiano)':      conf_dir / 'confusion_matrix_rf_estado_operacional.png',
    'XGBoost (Gaussiano)': conf_dir / 'confusion_matrix_xgboost_estado_operacional.png',
    'CNN-1D (FCN)':        conf_dir / 'confusion_matrix_cnn1d_estado_operacional.png',
}

for name, path in conf_files.items():
    if path.exists():
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.imshow(mpimg.imread(str(path)))
        ax.axis('off')
        ax.set_title(name, fontsize=13, pad=8)
        plt.tight_layout()
        plt.show()
    else:
        print(f'[{name}] Figura não encontrada: {path.name}')
        print('  Execute: python scripts/plot_confusion_matrix.py')

## 5.5 Comparação de Filtros — XGBoost

O impacto global da filtragem é pequeno (< 0,2 p.p. de F1-macro), mas o efeito **varia por classe**:
- Eventos de dinâmica lenta (Incrustação, Hidrato Trans.): **beneficiam** da filtragem
- Eventos de dinâmica rápida (DHSV, Golfadas): **perdem** com a suavização

In [ ]:
filter_names = {
    'Gaussiano':    'XGBoost (Gaussiano)',
    'Sem filtro':   'XGBoost (Sem filtro)',
    'Estatístico':  'XGBoost (Estatístico)',
}
filter_m = {k: metrics_raw[v] for k, v in filter_names.items()}

print('=== XGBoost — Impacto do Filtro (métricas globais) ===')
print(f'{"Filtro":<15} {"F1-macro":>10} {"F1-weighted":>12} {"Accuracy":>10}')
print('-' * 50)
for name, m in filter_m.items():
    mc = m['metrics_concat_folds']
    print(f'{name:<15} {mc["f1_macro"]:>10.4f} {mc["f1_weighted"]:>12.4f} {mc["accuracy"]:>10.4f}')

print('\n=== Efeito por classe (F1 — mostrando variação > 0.01) ===')
print(f'{"Cls":>4} {"Nome":<28} {"Gauss":>7} {"Sem":>7} {"Estat":>7} {"Melhor":>10}')
print('-' * 68)
all_cls = sorted({int(k) for m in filter_m.values() for k in m['per_class']})
for c in all_cls:
    str_c = str(c)
    f1s = {k: m['per_class'].get(str_c, {}).get('f1', float('nan')) for k, m in filter_m.items()}
    vals = [v for v in f1s.values() if not np.isnan(v)]
    if not vals or (max(vals) - min(vals)) < 0.01:
        continue
    best = max(f1s, key=lambda k: f1s[k] if not np.isnan(f1s[k]) else -1)
    label = WINDOW_CLASSES.get(c, str(c))[:26]
    print(f'{c:>4} {label:<28} {f1s["Gaussiano"]:>7.3f} {f1s["Sem filtro"]:>7.3f} '
          f'{f1s["Estatístico"]:>7.3f} {best:>10}')

## 5.6 CNN-1D vs XGBoost — Análise por Classe

A diferença global de 24 p.p. esconde comportamentos opostos:

**CNN-1D se aproxima do XGBoost:**
- Eventos ativos com padrão visual marcante em qualquer sensor (Golfadas, Hidrato Serviço, DHSV Ativo)

**CNN-1D fica muito abaixo:**
- Transientes — requerem `diff1_std`/`diff2_std` para detecção; CNN precisa aprender isso implicitamente
- Classes raras (PCK Incrustação) — poucos exemplos não são suficientes para aprendizado implícito

In [ ]:
xgb_pc   = metrics_raw['XGBoost (Gaussiano)']['per_class']
cnn_pc   = metrics_raw['CNN-1D (FCN)'].get('per_class_f1', {})
cnn_by_name = {v: cnn_pc.get(v, float('nan')) for v in WINDOW_CLASSES.values()}

print(f'{"Cls":>4} {"Nome":<32} {"XGBoost":>8} {"CNN-1D":>8} {"Delta":>7} {"Conclusão"}')
print('-' * 80)
for str_c, stats in sorted(xgb_pc.items(), key=lambda x: int(x[0])):
    c       = int(str_c)
    label   = WINDOW_CLASSES.get(c, str(c))
    f1_xgb  = stats['f1']
    f1_cnn  = cnn_by_name.get(label, float('nan'))
    delta   = f1_xgb - f1_cnn if not np.isnan(f1_cnn) else float('nan')
    cnn_str = f'{f1_cnn:.3f}' if not np.isnan(f1_cnn) else '  N/A'
    d_str   = f'{delta:+.3f}' if not np.isnan(delta) else '   N/A'
    if np.isnan(delta):
        conclusion = '—'
    elif delta < 0.05:
        conclusion = 'CNN competitiva'
    elif delta < 0.20:
        conclusion = 'XGBoost melhor'
    else:
        conclusion = 'XGBoost muito melhor'
    print(f'{c:>4} {label[:30]:<32} {f1_xgb:>8.3f} {cnn_str:>8} {d_str:>7} {conclusion}')